# How a decision model works: Laya for I-GUIDE AI

The I-GUIDE agent makes many small decisions on every request. For example: *does this search need the knowledge graph? which tool should run first?* Today an LLM makes each one by writing text. This notebook shows a different kind of model built for exactly this job, a **decision model**, using the open model [Laya](https://huggingface.co/convaiinnovations/laya):

1. How an LLM makes a decision today
2. How a decision model makes one instead, and how the two differ
3. Example queries: **vanilla Laya** (as downloaded) vs **fine-tuned Laya** (trained on I-GUIDE decisions)

**Requirements:** CPU is fine. You need about 4 GB of RAM and `pip install -r requirements.txt`. Both models are in `checkpoints/` (Git LFS); see the README.

To run both models over the full test sets, see `run_models.ipynb`.

In [1]:
import json, time, warnings
warnings.filterwarnings("ignore", message="IProgress not found")   # tqdm, without ipywidgets
from pathlib import Path

import pandas as pd
import torch
from laya.agent import Agent
from laya.common import build_sequence

pd.set_option("display.max_colwidth", 80)
VANILLA = Path("checkpoints/laya-vanilla")        # both checkpoints ship in this repo (Git LFS)
FINETUNED = Path("checkpoints/laya-finetuned")
for d in (VANILLA, FINETUNED):
    weights = d / "model.safetensors"
    assert weights.exists() and weights.stat().st_size > 10_000_000, (
        f"{weights} is missing or still a Git LFS pointer: run `git lfs pull` in the repo")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
QUESTIONS = json.loads(Path("data/questions.json").read_text(encoding="utf-8"))


def load(path):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")   # vanilla Laya clamps one out-of-range temperature, with a warning
        return Agent(str(path), device=DEVICE)


print("device:", DEVICE)

device: cuda


## 1 · How an LLM makes a decision today

The search router (`rag_pipeline/router_llm.py` in iguide-ai) sends a prompt like this to an LLM (gpt-5.6-luna):

```text
Decide which search methods to add for this query.
Query: most popular notebooks
Guidelines:
- spatial: enable when locations/coordinates/regions/distance/routing/maps are involved ...
- graph: enable when entity-relationship traversal ... helps (authors, organizations, tags, related resources)
Return STRICT JSON only: {"use_semantic": true|false, "use_spatial": true|false, "use_graph": true|false, "rationale": {...}}
```

The LLM **writes** a reply one token at a time, for example `{"use_semantic": false, "use_spatial": false, "use_graph": false, "rationale": {...}}`. Code then parses the JSON and reads off the two flags. That works, but:

- **The answer is free text.** The code has to hope it parses, and has to fall back to something when it doesn't.
- **There is no confidence.** You get `true` or `false`, never "probably".
- **It's slow and costs money.** About 1.6 s per decision over the API in our tests, and paid per call.
- **It can only be steered through the prompt.** This prompt never mentions popularity, so the LLM sends *most popular notebooks* to no graph search, even though the knowledge graph is where popularity lives.

## 2 · How a decision model makes a decision

A decision model never writes text. You give it:

- **a state:** the thing to judge, such as a search query;
- **questions** with a fixed set of answers.

It returns **a probability for every answer**, in a single pass through the network. Laya supports three question types:

| type | answers | returns |
|---|---|---|
| `noul` | yes / no | P(yes) |
| `choice` | one of K options | a probability for each option |
| `score` | ordered levels 0..K-1 | the expected level |

The router's decision becomes two yes/no questions:

In [2]:
ROUTER_Q = QUESTIONS["router_finetuned"]   # the wording the fine-tuned model was trained on
print(json.dumps(ROUTER_Q, indent=2))

{
  "graph": {
    "type": "noul",
    "instructions": "Does this query name a specific person, organization or tag, ask what is popular, or ask for resources linked to another resource?"
  },
  "spatial": {
    "type": "noul",
    "instructions": "Does this query mention a specific place, coordinates, or a distance or area on a map?"
  }
}


Inside Laya (421M parameters: a ModernBERT-large encoder plus a small decision head), each question is turned into **one token sequence**. The sequence holds the question, one `[MASK]` marker per possible answer, and the state:

```text
[CLS] noul question: <question> [SEP] [MASK] false: ... [MASK] true: ... [SEP] <query> [SEP]
```

The encoder reads the whole sequence at once. The head gives a score to the position of each `[MASK]`, and a softmax over those scores gives the probabilities. **The answers are positions in the input, not words to generate**, so the model can't answer anything outside the options. Here is the real sequence for one question:

In [3]:
vanilla = load(VANILLA)
query = "most popular notebooks"
ids, markers = build_sequence(vanilla.tok, query, Agent._to_internal(ROUTER_Q["graph"]),
                              vanilla.cfg["max_len"], vanilla.cfg["head_max_len"])
print(vanilla.tok.decode(ids, clean_up_tokenization_spaces=False))
print(f"\n{len(ids)} tokens; the two answers are scored at token positions {markers}")

[CLS]noul question: Does this query name a specific person, organization or tag, ask what is popular, or ask for resources linked to another resource?[SEP][MASK] false: no, the statement does not hold[MASK] true: yes, the statement holds[SEP]most popular notebooks[SEP]

56 tokens; the two answers are scored at token positions [32, 42]


And this is what comes back. The output is the whole answer, with nothing to parse. Note `output_tokens: 0`:

In [4]:
vanilla.system_one(query, ROUTER_Q)                       # warm-up: the first call pays one-time setup
t0 = time.perf_counter()
answer = vanilla.system_one(query, ROUTER_Q)
print(f"{(time.perf_counter() - t0) * 1000:.0f} ms on {DEVICE}\n")
print(json.dumps(answer, indent=2))

78 ms on cuda

{
  "model": "laya-rl-agent",
  "answers": {
    "graph": {
      "type": "noul",
      "noul": 0.1768,
      "confidence": 0.8232,
      "answer_confidence": 0.8232,
      "action": {
        "act_probability": 1.0
      }
    },
    "spatial": {
      "type": "noul",
      "noul": 0.1047,
      "confidence": 0.8953,
      "answer_confidence": 0.8953,
      "action": {
        "act_probability": 1.0
      }
    }
  },
  "usage": {
    "input_tokens": 105,
    "output_tokens": 0
  }
}


| | LLM (today) | Decision model (Laya) |
|---|---|---|
| Output | generated text, parsed afterwards | one probability per allowed answer |
| Can answer outside the options | yes | no |
| Confidence | none | a probability |
| Speed | ~1.6 s over the API | ~0.1 s on a GPU, ~0.5–1 s on a CPU |
| Cost and data | paid per call, sent to the provider | free, runs on your own machine |
| How to improve it | rewrite the prompt | fine-tune it on examples |

The catch is in the answer above. Vanilla Laya gives *most popular notebooks* P(graph) ≈ 0.18, so it says no, even though the question spells out "ask what is popular". Out of the box it doesn't know what I-GUIDE's decisions mean. That's what fine-tuning is for.

## 3 · Vanilla vs fine-tuned Laya on real I-GUIDE questions

**Fine-tuned Laya** is vanilla Laya trained for about an hour on one GPU, on two kinds of examples:
- **3,541 catalog search queries**, each labelled graph? and spatial?;
- **2,861 analysis requests**, each labelled with the tool that should run first.

An LLM wrote the requests and a second LLM labelled them against a written rubric. The training data is in `data/train/` and the script is `training/finetune.py`.

Below, both models answer two of the agent's real decisions:

1. **Search routing:** does a catalog query need the graph search, the spatial search, or both? Two yes/no questions.
2. **First tool:** which of the 53 analysis tools should run first for a request? One `choice` question with 53 options.

Each model is asked in the wording that suits it best. The fine-tuned model gets the exact wording it was trained on; vanilla Laya gets the wording that scored best for it in testing. The *gold* columns are hand labels from `data/eval/`.

In [5]:
ROUTER_EXAMPLES = ["most popular notebooks", "datasets maintained by NOAA", "publications by John Smith",
                   "weather stations near Chicago", "Sentinel-2 imagery over Nairobi",
                   "NOAA precipitation datasets near Houston", "how do I compute NDVI from Sentinel-2 bands?",
                   "machine learning for flood prediction"]
TOOL_EXAMPLES = ["Pull the boundary of Travis County, Texas.", "Grid the permits into 1 km squares and shade by count.",
                 "Look for datasets on wildfire smoke exposure.", "What does Moran's I measure?",
                 "How alike are central Paris and central Berlin from orbit?"]

read = lambda p: [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]
router_gold = {r["query"]: r for r in read("data/eval/router_eval_queries.jsonl")}
tool_gold = {r["request"]: r for r in read("data/eval/tool_eval_holdout.jsonl")}
TOOL_Q = {"tool": QUESTIONS["tool"]}
print(f"{len(QUESTIONS['tool']['criteria'])} tool options, e.g.:")
for name in ("admin_boundary", "aggregate_to_grid", "request_capability", "none"):
    print(f"  {name:20} {QUESTIONS['tool']['criteria'][name]}")


def ask_all(model, router_wording):
    # Every example through one model: P(yes) per router question, and the tool distribution.
    router = model.predict_batch(ROUTER_EXAMPLES, QUESTIONS[router_wording])
    tools = model.predict_batch([f"{tool_gold[r]['context']}\nRequest: {r}" for r in TOOL_EXAMPLES], TOOL_Q,
                                batch_size=2, max_len=1600, head_max_len=1400)   # 53 options need a bigger budget
    return {"router": [{f: a["answers"][f]["noul"] for f in ("graph", "spatial")} for a in router],
            "tools": [a["answers"]["tool"]["probabilities"] for a in tools]}


t0 = time.perf_counter()
results = {"vanilla": ask_all(vanilla, "router_zero_shot")}
del vanilla                                   # one model in memory at a time (~2.5 GB each on CPU)
results["fine-tuned"] = ask_all(load(FINETUNED), "router_finetuned")
print(f"\nboth models answered {len(ROUTER_EXAMPLES)} queries and {len(TOOL_EXAMPLES)} requests in {time.perf_counter() - t0:.0f}s")

53 tool options, e.g.:
  admin_boundary       THE tool for the boundary of a named US state, county or city —…
  aggregate_to_grid    Bin POINTS INTO A HEX OR SQUARE GRID and map the density as a…
  request_capability   Request another capability (search/analyze/code) when you cannot complete your task without it — e.g.
  none                 call no tool; answer directly from knowledge or the conversation



both models answered 8 queries and 5 requests in 5s


### Search routing: does the query need the graph search? the spatial search?

Each cell shows P(yes). ✗ marks an answer that disagrees with the gold label at the 0.5 cut-off.

In [6]:
def mark(p, gold):
    return f"{p:.2f}" + ("" if (p >= 0.5) == gold else " ✗")


rows = []
for i, q in enumerate(ROUTER_EXAMPLES):
    g = router_gold[q]
    row = {"query": q}
    for f in ("graph", "spatial"):
        row[(f, "gold")] = "yes" if g[f] else "no"
        for m in ("vanilla", "fine-tuned"):
            row[(f, m)] = mark(results[m]["router"][i][f], g[f])
    rows.append(row)
table = pd.DataFrame(rows).set_index("query")
table.columns = pd.MultiIndex.from_tuples(table.columns)
table

graph                    spatial  \
                                              gold vanilla fine-tuned    gold   
query                                                                           
most popular notebooks                         yes  0.07 ✗       0.99      no   
datasets maintained by NOAA                    yes  0.30 ✗       1.00      no   
publications by John Smith                     yes  0.30 ✗       1.00      no   
weather stations near Chicago                   no    0.01       0.00     yes   
Sentinel-2 imagery over Nairobi                 no    0.00       0.00     yes   
NOAA precipitation datasets near Houston       yes  0.00 ✗       0.99     yes   
how do I compute NDVI from Sentinel-2 bands?    no    0.00       0.00      no   
machine learning for flood prediction           no    0.00       0.00      no   

                                                                 
                                             vanilla fine-tuned  
query                                                            
most popular notebooks                          0.10       0.00  
datasets maintained by NOAA                     0.11       0.00  
publications by John Smith                      0.01       0.00  
weather stations near Chicago                 0.28 ✗       1.00  
Sentinel-2 imagery over Nairobi                 0.71       1.00  
NOAA precipitation datasets near Houston      0.25 ✗       1.00  
how do I compute NDVI from Sentinel-2 bands?    0.03       0.00  
machine learning for flood prediction           0.06       0.00

- **Vanilla Laya** is cautious about everything. Most probabilities stay below 0.5, so it misses the organization (*NOAA*), popularity (*most popular*) and several places. It isn't random, though: places do score higher than non-places. It just hasn't learned where I-GUIDE draws the line.
- **Fine-tuned Laya** gets every one of these right, including *most popular notebooks*, which the production LLM misses. And it is confident: almost every answer is close to 0 or 1.

### First tool: which of 53 tools should the analysis agent call first?

Each cell shows the model's top choice and its probability. ✓ means the choice is among the gold tools.

In [7]:
def top(probs, gold, n=2):
    best = sorted(probs.items(), key=lambda kv: -kv[1])[:n]
    return ("✓ " if best[0][0] in gold else "✗ ") + " · ".join(f"{k} {v:.2f}" for k, v in best)


rows = []
for i, r in enumerate(TOOL_EXAMPLES):
    g = tool_gold[r]
    rows.append({"request": r, "files attached": g["context"].removeprefix("Attached files: ")[:40],
                 "gold": ", ".join(g["gold"]),
                 "vanilla": top(results["vanilla"]["tools"][i], g["gold"]),
                 "fine-tuned": top(results["fine-tuned"]["tools"][i], g["gold"])})
pd.DataFrame(rows).set_index("request")

,files attached,gold,vanilla,fine-tuned
request,,,,
"Pull the boundary of Travis County, Texas.",none,admin_boundary,✗ add_map_layer 0.48 · cluster_points 0.09,✓ admin_boundary 1.00 · intersect_layers 0.00
Grid the permits into 1 km squares and shade by count.,permits.csv (points; columns: issued_dat,aggregate_to_grid,✗ add_map_layer 0.61 · embed_region 0.11,✓ aggregate_to_grid 0.99 · execute_code 0.00
Look for datasets on wildfire smoke exposure.,none,"request_capability, none",✗ add_map_layer 0.37 · none 0.32,✓ request_capability 0.99 · none 0.00
What does Moran's I measure?,none,none,✗ add_map_layer 0.68 · request_capability 0.05,✓ none 1.00 · global_spatial_autocorrelation 0.00
How alike are central Paris and central Berlin from orbit?,none,"compare_regions, geocode_places",✗ add_map_layer 0.64 · cluster_points 0.06,✗ none 0.94 · compare_regions 0.06


- **Vanilla Laya** answers `add_map_layer` to nearly everything. With 53 options and no training, it falls back on the most generic-sounding tool.
- **Fine-tuned Laya** picks the right tool for:
  - a named county with no file attached (`admin_boundary`);
  - a file of points to bin (`aggregate_to_grid`);
  - a literature search, which it hands to the supervisor (`request_capability`);
  - a conceptual question that needs no tool (`none`).
- **Its main failure mode shows in the last row.** An embedding comparison of two foreign cities comes back `none`, and confidently.

### Try your own

Add queries or requests and run the cell. Each model is loaded in turn, which takes about 10–15 s per model on CPU.

In [8]:
MY_QUERIES = ["trending flood datasets from NOAA for Louisiana"]                           # search-routing questions
MY_REQUESTS = [("Make a 500 m buffer around each school.", "schools.geojson (points; columns: name)")]   # (request, attached files or "none")

out = []
for name, ckpt, wording in (("vanilla", VANILLA, "router_zero_shot"), ("fine-tuned", FINETUNED, "router_finetuned")):
    model = load(ckpt)
    for q in MY_QUERIES:
        a = model.system_one(q, QUESTIONS[wording])["answers"]
        out.append({"input": q, "model": name, "answer": f"P(graph) {a['graph']['noul']:.2f} · P(spatial) {a['spatial']['noul']:.2f}"})
    for req, files in MY_REQUESTS:
        t = model.system_one(f"Attached files: {files}\nRequest: {req}", TOOL_Q, max_len=1600, head_max_len=1400)
        best = sorted(t["answers"]["tool"]["probabilities"].items(), key=lambda kv: -kv[1])[:3]
        out.append({"input": req, "model": name, "answer": " · ".join(f"{k} {v:.2f}" for k, v in best)})
    del model
pd.DataFrame(out).set_index(["input", "model"])

,,answer
input,model,
trending flood datasets from NOAA for Louisiana,vanilla,P(graph) 0.06 · P(spatial) 0.12
Make a 500 m buffer around each school.,vanilla,add_map_layer 0.67 · request_capability 0.09 · count_points_in_areas 0.05
trending flood datasets from NOAA for Louisiana,fine-tuned,P(graph) 0.99 · P(spatial) 1.00
Make a 500 m buffer around each school.,fine-tuned,buffer_layer 1.00 · execute_code 0.00 · admin_boundary 0.00


## What fine-tuning bought

From the full evaluation (`docs/evaluation.md`), on hand-labelled test sets the models never saw in training:

| | Vanilla Laya | Fine-tuned Laya | LLM baseline |
|---|---|---|---|
| Search routing, graph / spatial accuracy (87 queries) | 0.63 / 0.67 | **0.95 / 0.99** | 0.87 / 0.93 (gpt-5.6-luna) |
| First tool, exact match (42 held-out requests) | 0.10 | **0.79** | 0.67 (gpt-4o) |

Caveats:

- **The fine-tuned model is overconfident.** On the yes/no questions it answers close to 0 or 1 even when wrong, so its confidence can't yet be used to decide when to hand a request to the LLM. On the tool choice the confidence is more useful.
- **Typos are its weakest spot.**
- **The tool comparison favours Laya.** If "inspect the file first" also counts as right, gpt-4o scores 0.95. And the test requests listed file columns, which the production agent never sees.
- **One person wrote all the test labels.**

To retrain, run `python training/finetune.py` with the data in `data/train/`. It takes about 75 minutes on a 12 GB GPU. The README has the exact command.